# ArborGrid · 02 Prata (Requisito 06)

| Tabela | Chave (o que identifica uma linha) | Origem |
|---|---|---|
| `prata_arvores` | `detection_id` (uma foto analisada pelo modelo) | API ArborGrid (Visão Computacional) |
| `prata_clima` | `region_key` + `data_hora` (uma região numa hora) | Open-Meteo |
| `prata_denuncias` | `id_denuncia` | app ArborGrid (simulado) |

Passos: Bronze (texto JSON) → view tipada → regras de qualidade (`motivo_rejeicao`) →
lote limpo com **uma linha por chave** (`QUALIFY ROW_NUMBER()`) → **MERGE** na Prata.
Rode as células de MERGE **duas vezes**: na segunda, nada é inserido nem atualizado.

In [0]:
-- Horários das fontes estão no fuso de São Paulo
SET TIME ZONE 'America/Sao_Paulo';

key,value
spark.sql.session.timeZone,America/Sao_Paulo


## Tabelas da Prata

In [0]:
CREATE TABLE IF NOT EXISTS workspace.gs.prata_arvores (
  detection_id            STRING    COMMENT 'chave: uma foto (vista) analisada pelo modelo',
  geosampa_id             STRING    COMMENT 'árvore no cadastro da Prefeitura (GeoSampa)',
  image_id                STRING,
  region_key              STRING,
  distrito                STRING,
  subprefeitura           STRING,
  perspective             STRING,
  data_hora_captura       TIMESTAMP COMMENT 'data/hora do lote operacional da API',
  mes_captura_streetview  DATE      COMMENT 'mês da foto do Street View',
  ultima_vistoria_sisgau  DATE      COMMENT 'última vistoria oficial (nulo = sem registro)',
  latitude                DOUBLE,
  longitude               DOUBLE,
  triagem_foto            STRING    COMMENT 'CRITICO | PROXIMIDADE | SEGURO (desta foto)',
  triagem_arvore          STRING    COMMENT 'triagem consolidada da árvore (tree_macro_triage)',
  triagem_arvore_detalhe  STRING    COMMENT 'tree_consolidated_triage',
  conflict_category       STRING,
  confianca               DOUBLE,
  model_version           STRING,
  copa_px                 BIGINT,
  wire_px                 BIGINT,
  pole_px                 BIGINT,
  contact_direct_px       BIGINT,
  wind_buffer_px          BIGINT,
  carregado_em            TIMESTAMP,
  atualizado_em           TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.gs.prata_clima (
  region_key     STRING    COMMENT 'chave 1/2',
  data_hora      TIMESTAMP COMMENT 'chave 2/2 (hora local)',
  rajada_kmh     DOUBLE    COMMENT 'wind_gusts_10m',
  vento_kmh      DOUBLE    COMMENT 'wind_speed_10m',
  chuva_mm       DOUBLE    COMMENT 'precipitation',
  ds_coleta      DATE      COMMENT 'coleta que gerou o valor (vence a mais recente)',
  carregado_em   TIMESTAMP,
  atualizado_em  TIMESTAMP
);

CREATE TABLE IF NOT EXISTS workspace.gs.prata_denuncias (
  id_denuncia          STRING    COMMENT 'chave',
  data_hora_relato     TIMESTAMP COMMENT 'quando o morador registrou no celular',
  data_hora_envio      TIMESTAMP COMMENT 'quando chegou ao servidor',
  atraso_envio_min     BIGINT    COMMENT 'minutos entre relato e envio (falta de sinal)',
  latitude             DOUBLE    COMMENT 'GPS do celular (dado pessoal)',
  longitude            DOUBLE    COMMENT 'GPS do celular (dado pessoal)',
  region_key           STRING    COMMENT 'região mais próxima do relato',
  tipo                 STRING,
  foto_url             STRING,
  reportante_email     STRING    COMMENT 'dado pessoal: vira hash na Ouro',
  reportante_telefone  STRING    COMMENT 'dado pessoal: vira máscara na Ouro',
  ds_coleta            DATE,
  carregado_em         TIMESTAMP,
  atualizado_em        TIMESTAMP
);
-- Minimização (LGPD): o NOME do morador fica só na Bronze e não é levado para a Prata.
-- O google_maps_url das árvores também não vem: ele expõe o identificador bruto da foto.

## Árvores: texto JSON `items` → lista → uma linha por foto

In [0]:
CREATE OR REPLACE TEMP VIEW arvores_tipado AS
WITH paginas AS (
  SELECT carregado_em,
         from_json(items,
           'ARRAY<STRUCT<detection_id: STRING, image_id: STRING, geosampa_id: STRING,
             region_key: STRING, distrito: STRING, subprefeitura: STRING,
             sisgau_last_inspection: STRING, perspective: STRING, streetview_capture_month: STRING,
             captured_at: STRING, latitude: DOUBLE, longitude: DOUBLE,
             triage_class: STRING, tree_consolidated_triage: STRING, tree_macro_triage: STRING,
             conflict_category: STRING, confidence: DOUBLE, model_version: STRING,
             copa_px: BIGINT, wire_px: BIGINT, pole_px: BIGINT,
             contact_direct_px: BIGINT, wind_buffer_px: BIGINT>>') AS lista
  FROM workspace.gs.bronze_arvores
),
fotos AS (
  SELECT carregado_em, explode(lista) AS f FROM paginas  -- dias sem lote (lista vazia) não geram linhas
)
SELECT
  f.detection_id, f.geosampa_id, f.image_id, f.region_key, f.distrito, f.subprefeitura, f.perspective,
  to_timestamp(f.captured_at)                          AS data_hora_captura,
  to_date(concat(f.streetview_capture_month, '-01'))   AS mes_captura_streetview,
  to_date(substr(f.sisgau_last_inspection, 1, 10))     AS ultima_vistoria_sisgau,  -- '2013-06-12Z' -> 2013-06-12
  f.latitude, f.longitude,
  f.triage_class               AS triagem_foto,
  f.tree_macro_triage          AS triagem_arvore,
  f.tree_consolidated_triage   AS triagem_arvore_detalhe,
  f.conflict_category,
  f.confidence                 AS confianca,
  f.model_version, f.copa_px, f.wire_px, f.pole_px, f.contact_direct_px, f.wind_buffer_px,
  carregado_em
FROM fotos;

CREATE OR REPLACE TEMP VIEW arvores_validado AS
SELECT *,
  CASE
    WHEN detection_id IS NULL OR geosampa_id IS NULL OR data_hora_captura IS NULL
      THEN 'A1_chave_ou_data_nula'
    WHEN latitude IS NULL OR longitude IS NULL
      OR latitude NOT BETWEEN -24.01 AND -23.35 OR longitude NOT BETWEEN -46.83 AND -46.36
      THEN 'A2_localizacao_fora_de_sp'
    WHEN triagem_foto IS NULL OR triagem_foto NOT IN ('CRITICO', 'PROXIMIDADE', 'SEGURO')
      THEN 'A3_triagem_fora_do_catalogo'
    WHEN confianca IS NULL OR confianca < 0.7
      THEN 'A4_confianca_baixa'
  END AS motivo_rejeicao
FROM arvores_tipado;

## Clima: listas paralelas da Open-Meteo → uma linha por hora (`arrays_zip` + `explode`)

In [0]:
CREATE OR REPLACE TEMP VIEW clima_tipado AS
WITH base AS (
  SELECT region_key, CAST(ds_coleta AS DATE) AS ds_coleta, carregado_em,
         from_json(hourly,
           'STRUCT<time: ARRAY<STRING>, wind_gusts_10m: ARRAY<DOUBLE>,
                   wind_speed_10m: ARRAY<DOUBLE>, precipitation: ARRAY<DOUBLE>>') AS h
  FROM workspace.gs.bronze_clima
),
explodido AS (
  SELECT region_key, ds_coleta, carregado_em,
         explode(arrays_zip(h.time, h.wind_gusts_10m, h.wind_speed_10m, h.precipitation)) AS z
  FROM base
)
SELECT region_key,
       to_timestamp(z.time, "yyyy-MM-dd'T'HH:mm") AS data_hora,
       z.wind_gusts_10m  AS rajada_kmh,
       z.wind_speed_10m  AS vento_kmh,
       z.precipitation   AS chuva_mm,
       ds_coleta, carregado_em
FROM explodido;

CREATE OR REPLACE TEMP VIEW clima_validado AS
SELECT *,
  CASE
    WHEN region_key IS NULL OR data_hora IS NULL                      THEN 'C1_chave_nula'
    WHEN rajada_kmh IS NULL OR rajada_kmh < 0 OR rajada_kmh > 250      THEN 'C2_rajada_invalida'
    WHEN chuva_mm IS NULL OR chuva_mm < 0                              THEN 'C3_chuva_invalida'
  END AS motivo_rejeicao
FROM clima_tipado;

## Denúncias: tipagem, regras e região mais próxima do relato

In [0]:
CREATE OR REPLACE TEMP VIEW denuncias_tipado AS
SELECT id_denuncia,
       to_timestamp(data_hora_relato) AS data_hora_relato,
       to_timestamp(data_hora_envio)  AS data_hora_envio,
       CAST(latitude AS DOUBLE)       AS latitude,
       CAST(longitude AS DOUBLE)      AS longitude,
       tipo, foto_url,
       from_json(reportante, 'STRUCT<nome: STRING, email: STRING, telefone: STRING>') AS reportante,
       CAST(ds_coleta AS DATE)        AS ds_coleta,
       carregado_em
FROM workspace.gs.bronze_denuncias;

CREATE OR REPLACE TEMP VIEW denuncias_validado AS
SELECT *,
  CASE
    WHEN id_denuncia IS NULL OR data_hora_relato IS NULL OR data_hora_envio IS NULL
      THEN 'D1_chave_ou_data_nula'
    WHEN latitude IS NULL OR longitude IS NULL
      OR latitude NOT BETWEEN -24.01 AND -23.35 OR longitude NOT BETWEEN -46.83 AND -46.36
      THEN 'D2_sem_gps_ou_fora_de_sp'
    WHEN tipo IS NULL OR tipo NOT IN ('galho_na_fiacao', 'arvore_inclinada', 'arvore_seca', 'fio_partido')
      THEN 'D3_tipo_fora_do_catalogo'
    WHEN data_hora_envio < data_hora_relato
      THEN 'D4_envio_antes_do_relato'
  END AS motivo_rejeicao
FROM denuncias_tipado;

## Regras de qualidade: linhas barradas por regra (print para o relatório)

In [0]:
SELECT r.fonte, r.regra, r.motivo, COALESCE(c.barradas, 0) AS linhas_barradas
FROM VALUES
  ('arvores',   'A1_chave_ou_data_nula',       'foto sem identificador ou sem data não pode ser rastreada'),
  ('arvores',   'A2_localizacao_fora_de_sp',   'coordenada fora do município: a árvore não pode ir para o mapa'),
  ('arvores',   'A3_triagem_fora_do_catalogo', 'classe que o modelo não deveria produzir'),
  ('arvores',   'A4_confianca_baixa',          'confiança do modelo < 0,7: vai para revisão humana, não para o alerta'),
  ('clima',     'C1_chave_nula',               'sem região ou sem hora não dá para cruzar com as árvores'),
  ('clima',     'C2_rajada_invalida',          'rajada vazia, negativa ou > 250 km/h (fisicamente implausível)'),
  ('clima',     'C3_chuva_invalida',           'precipitação vazia ou negativa'),
  ('denuncias', 'D1_chave_ou_data_nula',       'denúncia sem identificador ou sem horário'),
  ('denuncias', 'D2_sem_gps_ou_fora_de_sp',    'celular sem GPS ou fora do município: não dá para localizar a árvore'),
  ('denuncias', 'D3_tipo_fora_do_catalogo',    'tipo de problema que o app não oferece'),
  ('denuncias', 'D4_envio_antes_do_relato',    'relógio do celular errado: horário não confiável')
  AS r(fonte, regra, motivo)
LEFT JOIN (
  SELECT 'arvores' AS fonte, motivo_rejeicao AS regra, COUNT(*) AS barradas
  FROM arvores_validado WHERE motivo_rejeicao IS NOT NULL GROUP BY motivo_rejeicao
  UNION ALL
  SELECT 'clima', motivo_rejeicao, COUNT(*)
  FROM clima_validado WHERE motivo_rejeicao IS NOT NULL GROUP BY motivo_rejeicao
  UNION ALL
  SELECT 'denuncias', motivo_rejeicao, COUNT(*)
  FROM denuncias_validado WHERE motivo_rejeicao IS NOT NULL GROUP BY motivo_rejeicao
) c USING (fonte, regra)
ORDER BY r.fonte, r.regra;

fonte,regra,motivo,linhas_barradas
arvores,A1_chave_ou_data_nula,foto sem identificador ou sem data não pode ser rastreada,0
arvores,A2_localizacao_fora_de_sp,coordenada fora do município: a árvore não pode ir para o mapa,0
arvores,A3_triagem_fora_do_catalogo,classe que o modelo não deveria produzir,0
arvores,A4_confianca_baixa,"confiança do modelo < 0,7: vai para revisão humana, não para o alerta",31
clima,C1_chave_nula,sem região ou sem hora não dá para cruzar com as árvores,0
clima,C2_rajada_invalida,"rajada vazia, negativa ou > 250 km/h (fisicamente implausível)",0
clima,C3_chuva_invalida,precipitação vazia ou negativa,0
denuncias,D1_chave_ou_data_nula,denúncia sem identificador ou sem horário,0
denuncias,D2_sem_gps_ou_fora_de_sp,celular sem GPS ou fora do município: não dá para localizar a árvore,8
denuncias,D3_tipo_fora_do_catalogo,tipo de problema que o app não oferece,8


## Lotes limpos: uma linha por chave

In [0]:
CREATE OR REPLACE TEMP VIEW lote_arvores AS
SELECT * EXCEPT (motivo_rejeicao), current_timestamp() AS atualizado_em
FROM arvores_validado
WHERE motivo_rejeicao IS NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY detection_id ORDER BY carregado_em DESC) = 1;

-- Cada coleta traz a previsão de ds até ds+2: a mesma hora aparece em até 3 coletas.
-- Fica a previsão mais recente (maior ds_coleta).
CREATE OR REPLACE TEMP VIEW lote_clima AS
SELECT * EXCEPT (motivo_rejeicao), current_timestamp() AS atualizado_em
FROM clima_validado
WHERE motivo_rejeicao IS NULL
QUALIFY ROW_NUMBER() OVER (PARTITION BY region_key, data_hora ORDER BY ds_coleta DESC, carregado_em DESC) = 1;

CREATE OR REPLACE TEMP VIEW lote_denuncias AS
WITH limpo AS (
  SELECT * EXCEPT (motivo_rejeicao)
  FROM denuncias_validado
  WHERE motivo_rejeicao IS NULL
  QUALIFY ROW_NUMBER() OVER (PARTITION BY id_denuncia ORDER BY carregado_em DESC) = 1
),
regioes AS (
  SELECT * FROM VALUES
    ('pinheiros', -23.5630, -46.6857), ('alto_da_mooca', -23.5616, -46.5928),
    ('vila_mariana', -23.5890, -46.6350), ('avenida_paulista', -23.5614, -46.6558)
  AS t(region_key, lat, lon)
),
mais_proxima AS (
  SELECT l.id_denuncia,
         min_by(g.region_key, pow(l.latitude - g.lat, 2) + pow(l.longitude - g.lon, 2)) AS region_key
  FROM limpo l CROSS JOIN regioes g
  GROUP BY l.id_denuncia
)
SELECT l.id_denuncia, l.data_hora_relato, l.data_hora_envio,
       CAST((unix_timestamp(l.data_hora_envio) - unix_timestamp(l.data_hora_relato)) / 60 AS BIGINT) AS atraso_envio_min,
       l.latitude, l.longitude, m.region_key, l.tipo, l.foto_url,
       l.reportante.email    AS reportante_email,
       l.reportante.telefone AS reportante_telefone,
       l.ds_coleta, l.carregado_em, current_timestamp() AS atualizado_em
FROM limpo l JOIN mais_proxima m USING (id_denuncia);

## MERGE (rode esta célula duas vezes: na 2ª, `num_inserted_rows` e `num_updated_rows` = 0)

In [0]:
MERGE INTO workspace.gs.prata_arvores t
USING lote_arvores s
ON t.detection_id = s.detection_id
WHEN MATCHED AND s.carregado_em > t.carregado_em THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
MERGE INTO workspace.gs.prata_clima t
USING lote_clima s
ON t.region_key = s.region_key AND t.data_hora = s.data_hora
WHEN MATCHED AND s.ds_coleta > t.ds_coleta THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


In [0]:
MERGE INTO workspace.gs.prata_denuncias t
USING lote_denuncias s
ON t.id_denuncia = s.id_denuncia
WHEN MATCHED AND s.carregado_em > t.carregado_em THEN UPDATE SET *
WHEN NOT MATCHED THEN INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


## Totais da Prata (print para o relatório)

In [0]:
SELECT 'prata_arvores' AS tabela, COUNT(*) AS linhas,
       COUNT(DISTINCT geosampa_id) AS arvores, MIN(data_hora_captura) AS de, MAX(data_hora_captura) AS ate
FROM workspace.gs.prata_arvores
UNION ALL
SELECT 'prata_clima', COUNT(*), COUNT(DISTINCT region_key), MIN(data_hora), MAX(data_hora)
FROM workspace.gs.prata_clima
UNION ALL
SELECT 'prata_denuncias', COUNT(*), COUNT(DISTINCT region_key), MIN(data_hora_relato), MAX(data_hora_relato)
FROM workspace.gs.prata_denuncias;

tabela,linhas,arvores,de,ate
prata_arvores,505,262,2026-11-10T08:05:21.000-03:00,2026-11-13T17:58:29.000-03:00
prata_clima,960,4,2026-11-10T00:00:00.000-03:00,2026-11-19T23:00:00.000-03:00
prata_denuncias,62,4,2026-11-10T06:04:00.000-03:00,2026-11-17T18:08:00.000-03:00
